# Scryfall (Magic: The Gathering) Card Analysis with Naive Bayes classifiers

In [ ]:
import pandas as pd

df = pd.read_json('data/oracle-cards.jsonl', lines=True)

## Data Preparation

In [ ]:
df.columns

In [ ]:
# Remove all land cards
df = df[~df['type_line'].str.contains('Land')]
df = df[['oracle_id', 'name', 'colors', 'oracle_text']]

df['colors'] = df['colors'].apply(lambda x: x if isinstance(x, list) else None)
df.dropna(subset=['colors'], inplace=True)

In [ ]:
df['colors'] = df['colors'].apply(lambda x: 'M' if len(x) > 1 else x[0] if x else 'C')
# Remove all multicolor cards
df = df[df['colors'] != "M"]
df.head(50)

In [ ]:
tag_df = pd.read_json('data/oracle-tags.jsonl', lines=True)

In [ ]:
tag_df.columns

In [ ]:
tag_df = tag_df[['label', 'taggings']]
tag_dict = {}
tag_labels = []
for index, row in tag_df.iterrows():
    taggings = row['taggings']
    label = row['label']
    card = False
    if isinstance(taggings, list):
        for i in range(len(taggings)):
            if(taggings[i]['oracle_id'] not in tag_dict):
                tag_dict[taggings[i]['oracle_id']] = []
            tag_dict[taggings[i]['oracle_id']].append(label)
            card = True
    if card:
        tag_labels.append(label)

In [ ]:
tag_dict

In [ ]:
tagging_row_indices = []
tagging_col_indices = []
id_list = df['oracle_id'].tolist()
for key, value in tag_dict.items():
    if key not in id_list:
        continue
    id_index = id_list.index(key)
    for label in value:
        tagging_row_indices.append(id_index)
        tagging_col_indices.append(tag_labels.index(label))

## Oracle Tag Analysis

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

data = np.ones(len(tagging_row_indices), dtype=int)
sparse_mat = csr_matrix(
    (data, (tagging_row_indices, tagging_col_indices)),
    shape=(len(df), len(tag_labels)),
    dtype=int
)

df_sparse = pd.DataFrame.sparse.from_spmatrix(sparse_mat, index=df.index, columns=tag_labels)

df_with_tags = pd.concat([df, df_sparse], axis=1)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

x_train, x_test, y_train_info, y_test_info = train_test_split(df_with_tags[tag_labels], df_with_tags[['colors', 'name', 'oracle_id']], test_size=0.2, random_state=42)

y_train = y_train_info['colors']
y_test = y_test_info['colors']

classifier = MultinomialNB()
classifier.fit(x_train, y_train)

y_pred = classifier.predict(x_test)
p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='weighted')
print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1-score: {f:.4f}")

### Confusion Matrix

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(y_test, y_pred)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=df['colors'].unique(), yticklabels=df['colors'].unique())
plt.xlabel('true label')
plt.ylabel('predicted label')

### Most confident correct and incorrect predictions for each color
#### Least confident prediction for each color

In [ ]:
y_pred_proba = classifier.predict_proba(x_test)
y_pred_conf = np.max(y_pred_proba, axis=1)
colors = df['colors'].unique()
for color in colors:
    correct_indices = np.where((y_pred == color) & (y_test == color))[0]
    if len(correct_indices) > 0:
        most_confident_index = correct_indices[np.argmax(y_pred_conf[correct_indices])]
        print(f"Most confident correct prediction for color {color}:")
        print(f"Predicted: {y_pred[most_confident_index]}, Actual: {y_test.iloc[most_confident_index]}, Confidence: {y_pred_conf[most_confident_index]}")
        print(f"Card name: {y_test_info.iloc[most_confident_index]['name']}")
        print(f"Tags: {tag_dict[y_test_info.iloc[most_confident_index]['oracle_id']]}")
        print()

for color in colors:
    incorrect_indices = np.where((y_pred == color) & (y_test != color))[0]
    if len(incorrect_indices) > 0:
        most_confident_index = incorrect_indices[np.argmax(y_pred_conf[incorrect_indices])]
        print(f"Most confident incorrect prediction for color {color}:")
        print(f"Predicted: {y_pred[most_confident_index]}, Actual: {y_test.iloc[most_confident_index]}, Confidence: {y_pred_conf[most_confident_index]}")
        print(f"Card name: {y_test_info.iloc[most_confident_index]['name']}")
        print(f"Tags: {tag_dict[y_test_info.iloc[most_confident_index]['oracle_id']]}")
        print()
        
for color in colors:
    colored_indices = np.where(y_test == color)[0]
    if len(colored_indices) > 0:
        least_confident_index = colored_indices[np.argmin(y_pred_conf[colored_indices])]
        print(f"Least confident prediction for color {color}:")
        print(f"Predicted: {y_pred[least_confident_index]}, Actual: {y_test.iloc[least_confident_index]}, Confidence: {y_pred_conf[least_confident_index]}")
        print(f"Card name: {y_test_info.iloc[least_confident_index]['name']}")
        if(y_test_info.iloc[least_confident_index]['oracle_id'] in tag_dict):
            print(f"Tags: {tag_dict[y_test_info.iloc[least_confident_index]['oracle_id']]}")
        else:
            print("Tags: None")
        print()


### Most relevant tag for each color

In [ ]:
for i, category in enumerate(classifier.classes_):
    print(f"Category: {category}")
    top_features_indices = np.argsort(classifier.feature_log_prob_[i])[-10:]
    top_features = [tag_labels[j] for j in top_features_indices]
    print(f"Top 10 features: {top_features}")
    print()

## Oracle Text Analysis

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

df['oracle_text'] = df['oracle_text'].fillna('').astype(str).str.replace("\n", " ")   # Fill NaN values with empty strings

x_train, x_test, y_train_info, y_test_info = train_test_split(df['oracle_text'], df[['colors', 'name', 'oracle_id']], test_size=0.2, random_state=42)
y_train = y_train_info['colors']
y_test = y_test_info['colors']

count_vect = CountVectorizer()
count_vect.fit(df['oracle_text'])

feature_names = count_vect.get_feature_names_out()
feature_names[1000:1009]

x_train_counts = count_vect.transform(x_train)
x_test_counts = count_vect.transform(x_test)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB()

classifier.fit(x_train_counts, y_train)

y_pred = classifier.predict(x_test_counts)

p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='weighted')
print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1-score: {f:.4f}")

### Most relevant words for each color

In [ ]:
feature_names = count_vect.get_feature_names_out()

for i, category in enumerate(classifier.classes_):
    print(f"Category: {category}")
    top_features_indices = np.argsort(classifier.feature_log_prob_[i])[-10:]
    top_features = [feature_names[j] for j in top_features_indices]
    print(f"Top 10 features: {top_features}")
    print()